In [1]:
import os
os.chdir('../pycode')

In [2]:
from modularpolynomials import *
from ecqf_tools import *
from nt import crt_pair,crt_list

In [14]:
aps_pc_ord = [ap for ap in get_aps_pc() if ap[0]>0]
ds_pc_ord = {a**2-4*p:[] for a,p in aps_pc_ord}
for a,p in aps_pc_ord:
    ds_pc_ord[a*a-4*p].append((a,p))

In [15]:
d_to_js = {}
for d in ds_pc_ord:
    for d0 in disc_closure(d):
        d_to_js[d0]={}

for ap in aps_pc_ord:
    a,p = ap
    jdata = {d:[] for d in disc_closure(a*a-4*p)}
    for j in ecqf_ord_1K_pc[(a,p)]:
        dj = qf_disc(ecqf_ord_1K_pc[(a,p)][j])
        jdata[dj].append(j)
    for d in jdata:
        d_to_js[d][p]=jdata[d]


In [18]:
d_to_js[-168]

{43: [10, 13, 15, 37],
 67: [20, 49, 61, 55],
 163: [49, 74, 83, 111],
 211: [69, 100, 108, 131],
 331: [163, 224, 327, 302],
 571: [63, 177, 73, 443],
 883: [5, 404, 663, 592],
 193: [186, 138, 12, 142],
 337: [153, 224, 99, 141],
 457: [14, 227, 412, 356],
 1009: [856, 1005, 961, 779],
 379: [145, 124, 179, 244],
 499: [497, 81, 128, 432],
 547: [458, 279, 323, 179],
 739: [489, 382, 106, 538],
 907: [743, 346, 193, 868],
 673: [121, 158, 357, 208]}

## Next steps
Right now, each discriminant has a dictionary of the form above - each key is a prime,
and the value associated to $p$ is the list  of j-invariants of elliptic curves mod $p$ whose endomorphism ring has discriminant equal to $d$. The lists all have the same size, since we are excluding supersingular classes.
1. Convert each list of j-invariants to the list of coefficients of the Hilbert polynomial - given $j_1,..,j_d$, compute $(x-j_1)...(x-j_d).
2. Arrange the coefficients in a matrix, so each row corresponds to a prime, and the $i$th column has the coefficients of $x^{i-1}$. For each column, we find the unique integer between $\pm\frac{p_1\cdots p_r}{2}$ that reduces to the correct coefficient mod each $p_i$.

Our set of primes is big enough if the largest coefficient of $H_d(x)$ has absolute value strictly smaller than $\frac{p_1\cdots p_r}{2}$.


## Implementation: `pycode/hilbert_crt.py`
The steps above are now implemented in `hilbert_crt.py`, on top of the new `PolyRing`/`Poly` classes in `alg_classes.py` (`poly_ring(base)` works over `ZZ`, `GF_p(p)`, `GF_pn`, and float `CC`; `from_roots` builds $(x-j_1)\cdots(x-j_h)$, `poly_crt` does the balanced coefficient-wise CRT).

The "enough primes" criterion: coefficients of $H_d$ are elementary symmetric functions of the CM $j$-values, so $\max|c_i| \le \prod_i (1+|j(\tau_i)|)$ with $\tau_i$ the roots of the reduced forms of disc $d$; the $j(\tau_i)$ are approximated from the Fourier expansion (used *only* for this bound). We are certified when $\sum \log_2 p_i > 1 + \sum \log_2(1+|j(\tau_i)|) + \text{safety}$. On the 81 known polynomials the bound is essentially exact (slack $\approx 0.1$ bits beyond the 8 safety bits).

In [ ]:
from hilbert_crt import *

data = cm_js_by_disc()          # {d: {p: roots of H_d mod p}}, consistency-checked
len(data)                       # distinct discriminants seen by the p<1024 data

In [ ]:
# the worked example from above, end to end
hilbert_via_crt(-168, data)

In [ ]:
# validation against the 81 known polynomials in hilbpolys.json:
# 69 certified + exact match, 12 uncertified (not enough primes below 1024 -- and
# indeed their all-primes CRT differs from the truth, so the criterion is honest)
check_against_known(data)

In [ ]:
# what can the current data certify? 120/1543 discs (max h = 5, down to d = -907);
# every new one was cross-checked against held-out primes not used in its CRT
rep = certification_report(data)
lib = build_hilbert_library(data)            # saved to data/hilbpolys_crt.json
sorted(d for d in lib)[:10], len(lib)

### What limits the library
The bound needs $\sum\log_2 p_i \gtrsim \sum_i \pi\sqrt{|d|}/(a_i\ln 2)$ bits, but each disc only has the primes $p<1024$ with $a^2-4p = dm^2$ — roughly 10 bits per prime and rarely more than ~20 primes. So growth requires **more/larger primes**: extending the bijection data past the 1024 cap (each new prime range certifies all its discs at once), and optionally the supersingular data (adds the ramified prime $p \mid d$ per disc). The 12 known-but-uncertified discs ($-71, -116, \dots, -284$) all sit 30–150 bits short.

## Beyond $p < 1024$: the search pipeline
Given a target $d$ and a ceiling $N$, `hilbert_poly_search(d, N)` either returns a fail message or computes $H_d$:

1. **`find_aps(d, N)`** — criterion 1: all $(p, a, m)$ with $p$ prime in $(1024, N]$ and $a^2-4p = m^2 d$, i.e. $d$ in the discriminant closure of $a^2-4p$. (The $m$-loop matters: $d \equiv 1 \bmod 8$ makes $m=1$ structurally empty.)
2. **`ap_status(a, p)`** — criterion 2: the class is usable when every prime $l \mid \mathrm{cond}(a^2-4p)$ has an Atkin polynomial (any depth). A non-Atkin $l \parallel c$ needs the ramified/ascending Vélu step (not implemented yet — flagged *fixable*); $l^2 \mid c$ without an Atkin polynomial is out of reach. For $N \approx 8000$ this almost never bites: a non-Atkin $m \ge 37$ forces $|d| \le 23$, and a non-Atkin prime in $\mathrm{cond}(d)$ forces $|d| \ge 3\cdot 37^2$.
3. **`class_endo_discs(a, p)`** — identification via *ancestor data alone* (`get_ancestor_data_ord`): leaves of the volcano carry disc $a^2-4p$ and each ancestor step divides the disc by $l^2$. No rigid l-set, no class-group labelling — so the 12 rigid-l-set gap discs (e.g. $D=-2624$) are fully usable now. Validated against the stored bijections on 905 classes (0 mismatches) and against class numbers on every call.
4. **`hilbert_search_report(d, N)`** — the bits budget: certifiable iff (bits on hand) + (bits from usable new primes) exceeds the coefficient bound.

The class scan (`trfr_to_js`) was the bottleneck — an $O(p^2)$ pile of powmods; `trace_frob` now uses a cached quadratic-character table + numpy, ~100×: a fresh class at $p \sim 6000$ costs ~0.2s. $H_{-152}$ rebuilds **from scratch** (zero precomputed data) in 0.5s; $H_{-971}$ ($h=15$, 420-bit coefficients, 36 primes up to 10343) in ~11s, hold-out-validated at $p \approx 60000$.

In [ ]:
# criterion 1 -- primes p in (1024, N] that see disc d:  a^2 - 4p = m^2 d
find_aps(-71, 4096)[:6]                       # (p, a, m), sorted by p

In [ ]:
# criterion 2 + budget: can we certify H_d from primes <= N?  (no classes computed)
rep = hilbert_search_report(-971, 40000)
rep['verdict'], rep['blocked'][:2]

In [ ]:
# end to end: fail message or the certified Hilbert polynomial
# (persists every harvested closure-disc group to hilbert_roots_ext.json)
rec = hilbert_poly_search(-971, 40000)
rec['certified'], rec['h'], len(rec['primes'])

## Step 3–4: $\Phi_p(X,Y)$ by CRT interpolation (`pycode/modpoly_crt.py`)
`phi_p_via_crt(p)` implements the plan: **general form** $a_{ij}(X^iY^j{+}X^jY^i)$, $0\le j\le i\le p$ (symmetry + monic of degree $p{+}1$ in $X$ + diagonal degree $2p$ force $a_{p+1,0}=1$, other $a_{p+1,j}=0$); **diagonal** $\Phi_p(X,X) = -\prod H_d^{m_d}$ over the support = union of closures of $a^2-4p$, $m_d = 1$ iff $p \mid d$ (degree identity $\sum m_d h(d) = 2p$ asserted); **interpolation mod $\ell$**: anti-diagonal sums from the char-0 diagonal + evaluations $\Phi_p(j_1,j_2)=0$ at $p$-isogenous pairs read off the CM-lattice side (ordinary + supersingular bijections at char $\ell$), Gaussian elimination over $\mathbb F_\ell$ (skip if underdetermined, *raise* if inconsistent); **CRT** with the Bröker–Sutherland height bound $\ln\|\Phi_p\| \le 6p\ln p + 16p + 14\sqrt p \ln p$, then a Kronecker-congruence check mod $p$.

**Pilot results** (vs the validated q-expansion cache): $p=5$: exact, certified, 0.9s, 27 primes. $p=7,11,13$: exact + certified. $p=17,19$: exact but uncertified (the B-S bound is ~1.5× the actual height; the $\ell<1024$ supply ran out). $p=23$: fails honestly (kronecker flag) — 68 usable primes supply 652 bits vs actual height 1025.

**Scaling wall**: a full solve mod $\ell$ needs $\gtrsim (p{+}1)(p{+}2)/2$ equations but pairs$(\ell) \sim \ell$, so $\ell \gtrsim p^2/2$ — at $p=37,53$ **zero** primes below 1024 qualify. Next moves: char-0 relations from special values (e.g. $\Phi_2(X,0) = -(X-54000)^3$ generalizes: $\Phi_p(X,j_0)$ for the 13 class-number-1 $j_0$'s factors into known Hilbert polynomials, ~$p$ relations each), joint solving across primes (an $\ell$ that pins only a subspace still contributes), vertical $p$-isogeny pairs, and bijections past 1024.

In [ ]:
from modpoly_crt import *

print('support(5):', diagonal_support(5))
print('Phi_5(X,X) =', phi_diagonal(5))
rec = phi_p_via_crt(5, verbose=False)
rec['certified'], rec['kronecker_ok'], check_phi_against_cache(5, rec['M'])